* ¿Hay nulos en client_id, transaction_date, amount? ¿Cuántos y por qué (¿son transacciones inválidas o clientes nuevos sin historial completo?)
* ¿Hay duplicados exactos de transacción?
* ¿El rango de fechas coincide con los 4.5 años esperados? ¿Hay huecos sospechosos (ej: un mes entero sin transacciones — señal de error de carga, no de negocio)?
* ¿Hay montos negativos o cero? (podrían ser devoluciones — ¿tu diseño las contempla o las descarta?)

In [24]:
from pathlib import Path
import json

import pandas as pd

In [25]:
BASE_DIR = Path("..")

DATA_RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_EDA_DIR = BASE_DIR / "outputs" / "m2" / "eda"

OUTPUT_EDA_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
# Carga datos
clients = pd.read_csv(DATA_RAW_DIR / "clients_raw.csv")
transactions = pd.read_csv(DATA_RAW_DIR / "transactions_raw.csv")
ground_truth = pd.read_csv(DATA_RAW_DIR / "_ground_truth.csv")

print(f"Clients shape: {clients.shape}")
print(f"Transactions shape: {transactions.shape}")
print(f"Ground Truth shape: {ground_truth.shape}")

Clients shape: (500, 7)
Transactions shape: (68483, 6)
Ground Truth shape: (500, 6)


In [27]:
print(clients.columns)
print(transactions.columns)
print(ground_truth.columns)

Index(['client_id', 'business_name', 'store_type', 'zone', 'latitude',
       'longitude', 'registration_date'],
      dtype='str')
Index(['transaction_id', 'client_id', 'date', 'amount', 'sku_count',
       'categories_purchased'],
      dtype='str')
Index(['client_id', 'archetype_initial', 'trajectory_type', 'inflection_month',
       'expected_cluster_tentative', 'expected_cluster_jun2025'],
      dtype='str')


In [28]:
# nulos
print("Nulos en clients:" , clients.isnull().sum())
print("Nulos en transactions:" , transactions.isnull().sum())
print("Nulos en ground_truth:" , ground_truth.isnull().sum())

Nulos en clients: client_id            0
business_name        0
store_type           0
zone                 0
latitude             0
longitude            0
registration_date    0
dtype: int64
Nulos en transactions: transaction_id          0
client_id               0
date                    0
amount                  0
sku_count               0
categories_purchased    0
dtype: int64
Nulos en ground_truth: client_id                       0
archetype_initial               0
trajectory_type                 0
inflection_month              189
expected_cluster_tentative      0
expected_cluster_jun2025      500
dtype: int64


In [29]:
# revisar duplicados
print("Duplicados en clients:" , clients.duplicated().sum())
print("Duplicados en transactions:" , transactions.duplicated().sum())


Duplicados en clients: 0
Duplicados en transactions: 0


In [30]:
# convertir fechas
clients['registration_date'] = pd.to_datetime(clients['registration_date'],errors='coerce')
transactions['date'] = pd.to_datetime(transactions['date'],errors='coerce') # coerce para convertir fechas inválidas a NaT
registration_date_invalidos = clients['registration_date'].isna().sum()
date_invalidos = transactions['date'].isna().sum()
print(f"Fechas inválidas en clients: {registration_date_invalidos}")
print(f"Fechas inválidas en transactions: {date_invalidos}")

# revisar rango de las fechas 
print('primera fecha de registro:', clients['registration_date'].min())
print('última fecha de registro:', clients['registration_date'].max())
print('primera fecha de transacción:', transactions['date'].min())
print('última fecha de transacción:', transactions['date'].max())

Fechas inválidas en clients: 0
Fechas inválidas en transactions: 0
primera fecha de registro: 2021-01-01 00:00:00
última fecha de registro: 2025-06-29 00:00:00
primera fecha de transacción: 2021-01-01 00:00:00
última fecha de transacción: 2025-06-30 00:00:00


In [31]:
# clientes cuya primera transacción ocurrió antes de su fecha de registro
primera_transaccion_por_cliente = (
    transactions
    .groupby("client_id", as_index=False)["date"]
    .min()
    .rename(columns={"date": "first_transaction_date"})
)

clientes_primera_transaccion_antes_registro = (
    primera_transaccion_por_cliente
    .merge(
        clients[["client_id", "business_name", "registration_date"]],
        on="client_id",
        how="inner",
    )
    .query("first_transaction_date < registration_date")
    .assign(
        dias_antes_registro=lambda df: (
            df["registration_date"] - df["first_transaction_date"]
        ).dt.days
    )
    .sort_values(["dias_antes_registro", "client_id"], ascending=[False, True])
)

print(
    "Clientes con primera transacción antes de registration_date:",
    len(clientes_primera_transaccion_antes_registro),
)

clientes_primera_transaccion_antes_registro


Clientes con primera transacción antes de registration_date: 340


,client_id,first_transaction_date,business_name,registration_date,dias_antes_registro
437,CLI-0440,2021-05-01,Bodega El Ahorro,2021-05-30,29
474,CLI-0477,2021-05-02,Minimarket Santa Rosa,2021-05-31,29
37,CLI-0038,2022-03-01,Bodega Los Andes,2022-03-29,28
102,CLI-0103,2021-04-02,Minimarket El Tambo,2021-04-30,28
175,CLI-0176,2021-03-02,Menú El Ahorro,2021-03-30,28
...,...,...,...,...,...
345,CLI-0348,2023-04-05,Bodega Santa Isabel,2023-04-06,1
382,CLI-0385,2022-05-13,Cevichería El Paraíso,2022-05-14,1
385,CLI-0388,2025-05-04,Bodega Don José,2025-05-05,1
473,CLI-0476,2021-09-02,Market San Martín,2021-09-03,1


In [37]:
# resumen EDA Bronze
client_duplicates = clients["client_id"].duplicated().sum()
transaction_duplicates = transactions["transaction_id"].duplicated().sum()
invalid_registration_dates = clients["registration_date"].isna().sum()
invalid_transaction_dates = transactions["date"].isna().sum()
amount_le_zero = (transactions["amount"] <= 0).sum()
sku_count_le_zero = (transactions["sku_count"] <= 0).sum()

clients_with_transactions = set(transactions["client_id"].unique()) & set(clients["client_id"].unique())
clients_without_transactions = set(clients["client_id"].unique()) - set(transactions["client_id"].unique())
transactions_without_master_client = set(transactions["client_id"].unique()) - set(clients["client_id"].unique())

eda_summary = {
    "n_clients": len(clients),
    "n_transactiones": len(transactions),
    "n_ground_truth": len(ground_truth),
    "client_id_duplicados": int(client_duplicates),
    "transactiones_id_duplicados": int(transaction_duplicates),
    "invalid_registration_dates": int(invalid_registration_dates),
    "invalid_transaction_dates": int(invalid_transaction_dates),
    "amount_le_zero": int(amount_le_zero),
    "sku_count_le_zero": int(sku_count_le_zero),
    "clients_con_transactions": len(clients_with_transactions),
    "clients_sin_transactions": len(clients_without_transactions),
    "clientes_primera_transaccion_antes_registro": len(clientes_primera_transaccion_antes_registro),
    "transactions_without_master_client": len(transactions_without_master_client),
    "transaction_date_min": str(transactions["date"].min().date()),
    "transaction_date_max": str(transactions["date"].max().date()),
    "registration_date_min": str(clients["registration_date"].min().date()),
    "registration_date_max": str(clients["registration_date"].max().date()),
}

eda_summary_path = OUTPUT_EDA_DIR / "eda_summary.json"
eda_summary_path.write_text(
    json.dumps(eda_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Resumen EDA guardado en: {eda_summary_path}")
eda_summary


Resumen EDA guardado en: ..\outputs\m2\eda\eda_summary.json


{'n_clients': 500,
 'n_transactiones': 68483,
 'n_ground_truth': 500,
 'client_id_duplicados': 0,
 'transactiones_id_duplicados': 0,
 'invalid_registration_dates': 0,
 'invalid_transaction_dates': 0,
 'amount_le_zero': 0,
 'sku_count_le_zero': 0,
 'clients_con_transactions': 498,
 'clients_sin_transactions': 2,
 'clientes_primera_transaccion_antes_registro': 340,
 'transactions_without_master_client': 0,
 'transaction_date_min': '2021-01-01',
 'transaction_date_max': '2025-06-30',
 'registration_date_min': '2021-01-01',
 'registration_date_max': '2025-06-29'}

In [32]:
# categorias unicas
unique_categories = set(transactions['categories_purchased'].dropna().str.split('|').explode())
sorted_unique_categories = sorted(unique_categories)
print(f"Categorías únicas en transactions: {sorted_unique_categories}")

Categorías únicas en transactions: ['agua', 'energeticas', 'gaseosas', 'jugos']


In [33]:
clients_con_transactions = set(transactions["client_id"].unique())
clients_master = set(clients["client_id"].unique())

clientes_sin_transactiones = sorted(
    clients_master - clients_con_transactions
)

clientes_sin_transactiones

['CLI-0313', 'CLI-0315']

In [34]:
compras_por_cliente = (
    transactions
    .groupby("client_id")
    .size()
    .reset_index(name="transaction_count")
    .sort_values("transaction_count", ascending=False)
)

print("Últimos 5 clientes con más compras:", compras_por_cliente.head())
print("Primeros 5 clientes con menos compras:", compras_por_cliente.tail())

Últimos 5 clientes con más compras:     client_id  transaction_count
495  CLI-0498                396
121  CLI-0122                391
168  CLI-0169                386
59   CLI-0060                382
199  CLI-0200                379
Primeros 5 clientes con menos compras:     client_id  transaction_count
463  CLI-0466                  2
339  CLI-0342                  1
228  CLI-0229                  1
156  CLI-0157                  1
80   CLI-0081                  1
